# 04 · Rama URL: XGBoost sobre features léxicas — Grupo 17
**Paso 5.** Entrena en PhiUSIIL (solo `URL` + etiqueta invertida) con `src/url_features.py`, que excluye esquema y `www`.

| Medición | Por qué importa |
|---|---|
| M1 · Artefacto: AUC usando SOLO esquema + www | Cuantifica la fuga de PhiUSIIL detectada en Q7 |
| M2 · PhiUSIIL test: AUC, TPR a FPR ≤ 1 % y 2 % | Desempeño dentro de la distribución |
| M3 · PhishTank 2008–2018 (solo positivos): tasa de detección | Validación externa y antigua |
| M4 · URLs de CORREOS reales (emails_v1): AUC por correo | **Desplazamiento de dominio (riesgo 2)**: la cifra que decide cuánto aporta la rama a la fusión |
| M5 · Latencia por URL | Insumo del KPI < 1 s |
| M6 · Variante B: URLs de correos, dominios disjuntos + fuente fuera | Rama URL principal si generaliza a dominios no vistos |

Salidas: `models/url_xgb_v1.json` + metadata, `data/processed/url_scores_v1.parquet` (puntaje por correo, para la fusión).
Las URLs se refangan **solo en memoria**; nada vivo se imprime ni se guarda.

In [8]:
# Celda 1: Entorno
import os, sys, json, random, time, importlib.util
from pathlib import Path
import numpy as np, pandas as pd
SEED = 17; random.seed(SEED); np.random.seed(SEED); os.environ["PYTHONHASHSEED"] = str(SEED)
def detectar_entorno():
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"): return "kaggle"
    try:
        import google.colab  # noqa
        return "colab"
    except ImportError:
        return "local"
ENV = detectar_entorno()
if ENV == "colab":
    from google.colab import drive; drive.mount("/content/drive"); BASE = Path("/content/drive/MyDrive/PI_G17")
elif ENV == "kaggle":
    BASE = Path("/kaggle/working/PI_G17")
else:
    BASE = Path(os.environ.get("PI_G17_BASE", Path.cwd() / "PI_G17"))
V1, COMP = BASE / "data/raw/g17-phishing-raw-v1", BASE / "data/raw/g17-phishing-complemento-v1"
PROC, MODELS, RES = BASE / "data/processed", BASE / "models", BASE / "results"
for p in (MODELS, RES): p.mkdir(parents=True, exist_ok=True)
def cargar(nombre, ruta):
    spec = importlib.util.spec_from_file_location(nombre, ruta); m = importlib.util.module_from_spec(spec)
    sys.modules[nombre] = m; spec.loader.exec_module(m); return m
ioc = cargar("ioc_utils", BASE / "src/ioc_utils.py"); uf = cargar("url_features", BASE / "src/url_features.py")
assert (PROC / "emails_v1.parquet").exists(), "Ejecuta primero 03_preparacion_splits.ipynb"
import xgboost as xgb, sklearn
print(ENV, "| xgboost", xgb.__version__, "| sklearn", sklearn.__version__)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
colab | xgboost 3.4.1 | sklearn 1.6.1


In [9]:
# Celda 2: PhiUSIIL: solo URL + etiqueta, deduplicado
P = pd.read_csv(next(V1.rglob("phiusiil_original.csv")), usecols=["URL", "label"])
P["y"] = 1 - P["label"]                                   # convención del proyecto: 1 = phishing
conflicto = P.groupby("URL")["y"].nunique()
P = P[~P["URL"].isin(conflicto[conflicto > 1].index)].drop_duplicates("URL").reset_index(drop=True)
t0 = time.time()
F = pd.DataFrame([uf.features_url(u) for u in P["URL"]])
print(f"{len(P)} URLs únicas | features en {time.time()-t0:.0f}s | phishing {P['y'].mean():.1%} | URLs con etiqueta en conflicto: {int((conflicto > 1).sum())}")

235370 URLs únicas | features en 19s | phishing 42.7% | URLs con etiqueta en conflicto: 0


In [10]:
# Celda 3: Split, codificación de TLD y M1 (artefacto)
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, roc_curve
idx_tr, idx_tmp = train_test_split(P.index, test_size=0.30, stratify=P["y"], random_state=SEED)
idx_va, idx_te = train_test_split(idx_tmp, test_size=0.50, stratify=P.loc[idx_tmp, "y"], random_state=SEED)
TOP_TLD = F.loc[idx_tr, "tld"].value_counts().head(50).index.tolist()   # aprendido SOLO de train
def preparar(Fx):
    X = Fx[uf.NUMERICAS].astype(float).copy()
    X["tld"] = pd.Categorical(Fx["tld"].where(Fx["tld"].isin(TOP_TLD), "__otro__"), categories=TOP_TLD + ["__otro__"])
    return X
X = preparar(F); y = P["y"].values
# M1: ¿cuánto separa la etiqueta un modelo que SOLO ve el artefacto?
A = np.c_[P["URL"].str.match(r"(?i)https://").astype(int), P["URL"].str.contains(r"(?i)://www\.").astype(int)]
lr_art = LogisticRegression().fit(A[idx_tr], y[idx_tr])
M1 = {"auc_solo_esquema_y_www": round(roc_auc_score(y[idx_te], lr_art.predict_proba(A[idx_te])[:, 1]), 4),
      # ¿las legítimas son páginas de inicio? Si sí, "ruta vacía" es otro artefacto aunque se excluya www/esquema
      "pct_ruta_vacia_por_clase": {int(k): round(100 * float(v), 1) for k, v in
                                   ((F["path_len"] <= 1) & (F["query_len"] == 0)).groupby(P["y"]).mean().items()},
      "auc_solo_path_len": round(roc_auc_score(y[idx_te], F.loc[idx_te, "path_len"] + F.loc[idx_te, "query_len"]), 4)}
M1

{'auc_solo_esquema_y_www': np.float64(0.9877),
 'pct_ruta_vacia_por_clase': {0: 100.0, 1: 71.8},
 'auc_solo_path_len': np.float64(0.7998)}

In [11]:
# Celda 4: XGBoost con early stopping y M2
modelo = xgb.XGBClassifier(n_estimators=2000, learning_rate=0.05, max_depth=8, subsample=0.8, colsample_bytree=0.8,
                           tree_method="hist", enable_categorical=True, eval_metric="aucpr",
                           early_stopping_rounds=50, random_state=SEED, n_jobs=-1)
t0 = time.time()
modelo.fit(X.loc[idx_tr], y[idx_tr], eval_set=[(X.loc[idx_va], y[idx_va])], verbose=False)
def tpr_a_fpr(yv, s, fpr_max):
    fpr, tpr, thr = roc_curve(yv, s); i = np.where(fpr <= fpr_max)[0][-1]; return float(tpr[i]), float(thr[i])
s_va = modelo.predict_proba(X.loc[idx_va])[:, 1]; s_te = modelo.predict_proba(X.loc[idx_te])[:, 1]
_, UMBRAL_FPR2 = tpr_a_fpr(y[idx_va], s_va, 0.02)            # umbral fijado en VALIDACIÓN
M2 = {"mejor_iteracion": int(modelo.best_iteration), "entrenamiento_s": round(time.time() - t0),
      "auc_test": round(roc_auc_score(y[idx_te], s_te), 4),
      "tpr_test_a_fpr_1pct": round(tpr_a_fpr(y[idx_te], s_te, 0.01)[0], 4),
      "tpr_test_a_fpr_2pct": round(tpr_a_fpr(y[idx_te], s_te, 0.02)[0], 4),
      "umbral_val_fpr2": round(UMBRAL_FPR2, 4),
      "tpr_fpr_test_con_umbral_val": {"tpr": round(float((s_te[y[idx_te] == 1] >= UMBRAL_FPR2).mean()), 4),
                                      "fpr": round(float((s_te[y[idx_te] == 0] >= UMBRAL_FPR2).mean()), 4)}}
imp = pd.Series(modelo.get_booster().get_score(importance_type="gain")).sort_values(ascending=False).head(8).round(1)
M2["top_features_gain"] = imp.to_dict(); M2

{'mejor_iteracion': 209,
 'entrenamiento_s': 9,
 'auc_test': np.float64(0.9319),
 'tpr_test_a_fpr_1pct': 0.7767,
 'tpr_test_a_fpr_2pct': 0.7918,
 'umbral_val_fpr2': 0.4845,
 'tpr_fpr_test_con_umbral_val': {'tpr': 0.791, 'fpr': 0.0189},
 'top_features_gain': {'path_len': 4381.7,
  'n_segmentos_path': 72.5,
  'tld_len': 59.4,
  'n_puntos_host': 56.0,
  'n_digitos_host': 54.5,
  'tld': 51.4,
  'n_palabras_sensibles': 36.7,
  'ratio_digitos': 31.9}}

In [12]:
# Celda 5: M3 PhishTank externo y M5 latencia
PT = pd.read_csv(COMP / "phishtank_github" / "phishtank.csv", usecols=["url"]).drop_duplicates()
PT = PT[~PT["url"].isin(set(P["URL"]))]                      # fuera lo que ya estaba en PhiUSIIL
s_pt = modelo.predict_proba(preparar(pd.DataFrame([uf.features_url(u) for u in PT["url"]])))[:, 1]
M3 = {"urls": int(len(PT)), "deteccion_con_umbral_val_fpr2": round(float((s_pt >= UMBRAL_FPR2).mean()), 4)}
muestra = P["URL"].sample(500, random_state=SEED).tolist()
t0 = time.perf_counter()
for u in muestra:
    modelo.predict_proba(preparar(pd.DataFrame([uf.features_url(u)])))
M5 = {"ms_por_url_individual": round(1000 * (time.perf_counter() - t0) / len(muestra), 2)}
M3, M5

({'urls': 32091, 'deteccion_con_umbral_val_fpr2': 0.9892},
 {'ms_por_url_individual': 6.15})

In [13]:
# Celda 6: M4 URLs de correos reales (desplazamiento de dominio)
D = pd.read_parquet(PROC / "emails_v1.parquet", columns=["id", "clase", "split", "grupo", "urls_defang"])
MAX_URLS = 20                                                  # tope por correo (declarado)
filas = [(i, ioc.refang(u)) for i, lst in zip(D["id"], D["urls_defang"]) for u in list(lst)[:MAX_URLS]]
U = pd.DataFrame(filas, columns=["id", "url"])                  # URLs vivas SOLO en memoria
t0 = time.time()
FU = pd.DataFrame([uf.features_url(u) for u in U["url"]])
U["host"] = [uf.descomponer(u)[0] for u in U["url"]]            # solo en memoria, para medir solapamiento
U = U.drop(columns="url")                                       # se descarta la forma viva de inmediato
U["s"] = modelo.predict_proba(preparar(FU))[:, 1] if len(U) else []
agg = U.groupby("id")["s"].agg(url_max="max", url_media="mean", n_urls_eval="size").reset_index()
S = D.drop(columns="urls_defang").merge(agg, on="id", how="left")
S["n_urls_eval"] = S["n_urls_eval"].fillna(0).astype(int)
def auc_seguro(a, b):
    y_ = np.r_[np.ones(len(a)), np.zeros(len(b))]; s_ = np.r_[a, b]
    return round(roc_auc_score(y_, s_), 4) if len(a) and len(b) else None
con = S[S["n_urls_eval"] > 0]
M4 = {"urls_puntuadas": int(len(U)), "segundos": round(time.time() - t0),
      "pct_correos_con_url_por_clase": S.groupby("clase")["n_urls_eval"].apply(lambda s: round(100 * (s > 0).mean(), 1)).to_dict()}
for sp in ("test", "todos"):
    c = con if sp == "todos" else con[con["split"] == "test"]
    ph = c.loc[c.clase == "phishing", "url_max"]
    M4[sp] = {"auc_phishing_vs_legitimo": auc_seguro(ph, c.loc[c.clase == "legitimo", "url_max"]),
              "auc_phishing_vs_spam": auc_seguro(ph, c.loc[c.clase == "spam", "url_max"]),
              "tpr_phishing_umbral_val": round(float((ph >= UMBRAL_FPR2).mean()), 4) if len(ph) else None,
              "fpr_legitimo_umbral_val": round(float((c.loc[c.clase == "legitimo", "url_max"] >= UMBRAL_FPR2).mean()), 4)}
M4["auc_phishing_vs_legitimo_por_grupo_de_phishing"] = {
    g: auc_seguro(con.loc[con.grupo == g, "url_max"], con.loc[con.clase == "legitimo", "url_max"])
    for g in sorted(con.loc[con.clase == "phishing", "grupo"].unique())}
M4

{'urls_puntuadas': 152364,
 'segundos': 8,
 'pct_correos_con_url_por_clase': {'legitimo': 71.5,
  'phishing': 73.1,
  'spam': 67.0},
 'test': {'auc_phishing_vs_legitimo': np.float64(0.6707),
  'auc_phishing_vs_spam': np.float64(0.7643),
  'tpr_phishing_umbral_val': 0.9551,
  'fpr_legitimo_umbral_val': 0.9591},
 'todos': {'auc_phishing_vs_legitimo': np.float64(0.6811),
  'auc_phishing_vs_spam': np.float64(0.7702),
  'tpr_phishing_umbral_val': 0.9615,
  'fpr_legitimo_umbral_val': 0.9563},
 'auc_phishing_vs_legitimo_por_grupo_de_phishing': {'Nigerian_Fraud': np.float64(0.3394),
  'nazario_anual': np.float64(0.737),
  'phishing_pot': np.float64(0.7603)}}

In [14]:
# Celda 6b: Variante B: URLs de CORREOS con DOMINIOS DISJUNTOS
# Etiqueta débil: cada URL hereda la clase de su correo. En la corrida anterior el 89,9 % de los hosts de test ya
# estaba en train: el AUC medía memoria de dominios. Ahora ningún dominio de val/test entra al entrenamiento.
# "dominio" = últimas 2 etiquetas del host (sin lista de sufijos públicos: limitación declarada).
U = U.merge(D[["id", "clase", "split", "grupo"]], on="id")
U["y"] = (U["clase"] == "phishing").astype(int)
U["dominio"] = U["host"].str.split(".").str[-2:].str.join(".")
XU = preparar(FU.loc[U.index])
def entrenar_b(m_tr, m_va):
    """None si train o val quedan con una sola clase tras excluir dominios (se registra, no se oculta)."""
    if U.loc[m_tr, "y"].nunique() < 2 or U.loc[m_va, "y"].nunique() < 2:
        return None
    mb = xgb.XGBClassifier(n_estimators=1000, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8,
                           tree_method="hist", enable_categorical=True, eval_metric="aucpr",
                           early_stopping_rounds=50, random_state=SEED, n_jobs=-1)
    mb.fit(XU[m_tr], U.loc[m_tr, "y"], eval_set=[(XU[m_va], U.loc[m_va, "y"])], verbose=False); return mb
dom_eval = set(U.loc[U["split"].isin(["val", "test"]), "dominio"])
m_tr = ((U["split"] == "train") & ~U["dominio"].isin(dom_eval)).values
m_va = (U["split"] == "val").values
modelo_b = entrenar_b(m_tr, m_va)
assert modelo_b is not None, "Con dominios disjuntos, train quedó con una sola clase: revisar antes de seguir"
U["s_b"] = modelo_b.predict_proba(XU)[:, 1]
agg_b = U.groupby("id")["s_b"].agg(url_b_max="max", url_b_media="mean").reset_index()
S = S.merge(agg_b, on="id", how="left")
c = S[(S["n_urls_eval"] > 0) & (S["split"] == "test")]
M6 = {"urls_train_disjuntas": int(m_tr.sum()), "urls_train_descartadas_por_dominio": int(((U["split"] == "train").values & ~m_tr).sum()),
      "mejor_iteracion": int(modelo_b.best_iteration),
      "test_auc_phishing_vs_legitimo": auc_seguro(c.loc[c.clase == "phishing", "url_b_max"], c.loc[c.clase == "legitimo", "url_b_max"]),
      "test_auc_phishing_vs_spam": auc_seguro(c.loc[c.clase == "phishing", "url_b_max"], c.loc[c.clase == "spam", "url_b_max"]),
      "test_auc_por_grupo_vs_legitimo": {g: auc_seguro(c.loc[c.grupo == g, "url_b_max"], c.loc[c.clase == "legitimo", "url_b_max"])
                                         for g in sorted(c.loc[c.clase == "phishing", "grupo"].unique())}}
# Fuente de phishing fuera: se entrena sin ese grupo y sin sus dominios; se evalúa ese grupo contra el legítimo de test
M6["loso_auc_vs_legitimo_test"] = {}
leg_test_ids = set(c.loc[c.clase == "legitimo", "id"])
for g in sorted(U.loc[U["clase"] == "phishing", "grupo"].unique()):
    dom_g = set(U.loc[U["grupo"] == g, "dominio"]) | dom_eval
    mtr = ((U["split"] == "train") & (U["grupo"] != g) & ~U["dominio"].isin(dom_g)).values
    mva = ((U["split"] == "val") & (U["grupo"] != g)).values
    mg = entrenar_b(mtr, mva)
    if mg is None:
        M6["loso_auc_vs_legitimo_test"][g] = "omitido: una sola clase tras excluir dominios"; continue
    sg = pd.Series(mg.predict_proba(XU)[:, 1], index=U.index)
    mx = sg.groupby(U["id"]).max()
    M6["loso_auc_vs_legitimo_test"][g] = auc_seguro(mx[mx.index.isin(set(U.loc[U["grupo"] == g, "id"]))].values,
                                                    mx[mx.index.isin(leg_test_ids)].values)
U = U.drop(columns=["host", "dominio"])
M6

# Celda 7: Guardar y RESUMEN
modelo.save_model(MODELS / "url_xgb_v1.json")
meta = {"version": "url_xgb_v1", "features_numericas": uf.NUMERICAS, "tld_categorias": TOP_TLD + ["__otro__"],
        "umbral_val_fpr2": UMBRAL_FPR2, "excluidas_por_artefacto": ["esquema", "www", "FILENAME", "URLSimilarityIndex", "features de página"],
        "xgboost": xgb.__version__, "semilla": SEED, "max_urls_por_correo": MAX_URLS}
(MODELS / "url_xgb_v1.meta.json").write_text(json.dumps(meta, indent=1, ensure_ascii=False))
modelo_b.save_model(MODELS / "url_xgb_v1b_correos.json")
S[["id", "url_max", "url_media", "url_b_max", "url_b_media", "n_urls_eval"]].to_parquet(PROC / "url_scores_v1.parquet", index=False)
R = {"entorno": ENV, "M1_artefacto": M1, "M2_phiusiil": M2, "M3_phishtank": M3, "M4_correos": M4, "M5_latencia": M5, "M6_variante_correos": M6}
(RES / "rama_url_v1.json").write_text(json.dumps(R, indent=1, ensure_ascii=False, default=str))
print("===== RESUMEN (04) =====")
print(json.dumps(R, ensure_ascii=False, default=str, separators=(",", ":")))

===== RESUMEN (04) =====
{"entorno":"colab","M1_artefacto":{"auc_solo_esquema_y_www":0.9877,"pct_ruta_vacia_por_clase":{"0":100.0,"1":71.8},"auc_solo_path_len":0.7998},"M2_phiusiil":{"mejor_iteracion":209,"entrenamiento_s":9,"auc_test":0.9319,"tpr_test_a_fpr_1pct":0.7767,"tpr_test_a_fpr_2pct":0.7918,"umbral_val_fpr2":0.4845,"tpr_fpr_test_con_umbral_val":{"tpr":0.791,"fpr":0.0189},"top_features_gain":{"path_len":4381.7,"n_segmentos_path":72.5,"tld_len":59.4,"n_puntos_host":56.0,"n_digitos_host":54.5,"tld":51.4,"n_palabras_sensibles":36.7,"ratio_digitos":31.9}},"M3_phishtank":{"urls":32091,"deteccion_con_umbral_val_fpr2":0.9892},"M4_correos":{"urls_puntuadas":152364,"segundos":8,"pct_correos_con_url_por_clase":{"legitimo":71.5,"phishing":73.1,"spam":67.0},"test":{"auc_phishing_vs_legitimo":0.6707,"auc_phishing_vs_spam":0.7643,"tpr_phishing_umbral_val":0.9551,"fpr_legitimo_umbral_val":0.9591},"todos":{"auc_phishing_vs_legitimo":0.6811,"auc_phishing_vs_spam":0.7702,"tpr_phishing_umbral_val